In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [3]:
import os, csv, json, random, numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
from collections import Counter, defaultdict

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
 
FEATURES_DIR = "/kaggle/input/datasets/ptrnghieu/hi-ef-features-v2"
DATASET_DIR = "/kaggle/input/datasets/ptrnghieu/hi-ef-dataset"
ZIP1 = os.path.join(DATASET_DIR, "Hi-EF-20260829T071606Z-1-001", "Hi-EF")
 
annotations = {}
with open(os.path.join(ZIP1, "annotation.csv"), 'r') as f:
    for row in csv.reader(f):
        if len(row) >= 2:
            cid = row[0].strip()
            annotations[cid] = {
                'text': row[1].strip() if len(row) > 1 else '',
                'scene_type': row[4].strip() if len(row) > 4 and row[4].strip() else None,
                'polarity': row[5].strip() if len(row) > 5 and row[5].strip() else None,
                'intensity': row[6].strip() if len(row) > 6 and row[6].strip() else None,
                'emotion': row[7].strip() if len(row) > 7 and row[7].strip() else None,
                'uncertainty': row[8].strip() if len(row) > 8 and row[8].strip() else None,
            }
 
samples = []
with open(os.path.join(ZIP1, "sample.csv"), 'r') as f:
    reader = csv.reader(f); next(reader)
    for row in reader: samples.append(row)
 
emotion_map = {'angry':0,'disgust':1,'fear':2,'happy':3,'neutral':4,'sad':5,'surprise':6}
scene_types = sorted(set(a.get('scene_type') for a in annotations.values() if a.get('scene_type')))
scene_map = {s:i for i,s in enumerate(scene_types)}
emo_names = ['angry','disgust','fear','happy','neutral','sad','surprise']
 
mcis_index = []
for s in samples:
    clips = [s[i].strip() for i in range(1, 5)]
    entry = {'sample_id': s[0].strip(), 'clip_ids': clips,
             'feature_files': [c.replace('/','_')+'.pt' for c in clips]}
    for ln, ci in [('clip3',2),('clip4',3)]:
        cid = clips[ci]
        if cid in annotations and annotations[cid].get('emotion'):
            ann = annotations[cid]
            entry[f'{ln}_emotion'] = emotion_map.get(ann['emotion'],-1)
            entry[f'{ln}_scene'] = scene_map.get(ann.get('scene_type',''),-1)
        else:
            entry[f'{ln}_emotion'] = -1; entry[f'{ln}_scene'] = -1
    mcis_index.append(entry)
 
def split_mcis(mcis_index, train_ratio=0.7, val_ratio=0.15, seed=42):
    rng = random.Random(seed)
    clip4_groups = {}
    for idx, e in enumerate(mcis_index):
        c4 = e['clip_ids'][3]
        clip4_groups.setdefault(c4, []).append(idx)
    keys = list(clip4_groups.keys()); rng.shuffle(keys)
    n = len(keys); nt = int(n*train_ratio); nv = int(n*val_ratio)
    return ([i for g in keys[:nt] for i in clip4_groups[g]],
            [i for g in keys[nt:nt+nv] for i in clip4_groups[g]],
            [i for g in keys[nt+nv:] for i in clip4_groups[g]])
 
train_idx, val_idx, test_idx = split_mcis(mcis_index)
print(f"Train: {len(train_idx)}, Val: {len(val_idx)}, Test: {len(test_idx)}")

Device: cuda
Train: 1980, Val: 424, Test: 426


In [4]:
N_CLASSES = 7
 
# Count A→B transitions in training data
trans_counts = np.zeros((N_CLASSES, N_CLASSES))
for idx in train_idx:
    e = mcis_index[idx]
    a, b = e['clip3_emotion'], e['clip4_emotion']
    if a >= 0 and b >= 0:
        trans_counts[a][b] += 1
 
# Normalize per row → P(B=j | A=i)
row_sums = trans_counts.sum(axis=1, keepdims=True)
row_sums[row_sums == 0] = 1
trans_prob = trans_counts / row_sums
 
# Also compute global class priors
global_counts = trans_counts.sum(axis=0)
global_prior = global_counts / global_counts.sum()
 
print("Empirical A→B transition matrix:")
header = f"{'A\\B':>10}" + "".join(f"{emo_names[j][:7]:>8}" for j in range(7))
print(header)
for i in range(7):
    row = f"{emo_names[i]:>10}"
    for j in range(7):
        row += f"{trans_prob[i][j]*100:>7.1f}%"
    row += f"  (n={int(trans_counts[i].sum())})"
    print(row)
 
print(f"\nGlobal prior: {[f'{emo_names[i]}:{global_prior[i]*100:.1f}%' for i in range(7)]}")
 
# Convert to tensors
T_MATRIX = torch.tensor(trans_prob, dtype=torch.float32).to(DEVICE)
G_PRIOR = torch.tensor(global_prior, dtype=torch.float32).to(DEVICE)

Empirical A→B transition matrix:
       A\B   angry disgust    fear   happy neutral     sad surpris
     angry   34.5%   12.5%    2.4%   12.3%   11.8%   17.0%    9.5%  (n=423)
   disgust   27.8%   15.4%    1.2%   22.8%   16.7%    8.0%    8.0%  (n=162)
      fear   24.0%   12.0%    4.0%   16.0%   12.0%   16.0%   16.0%  (n=25)
     happy   12.0%    7.3%    1.6%   46.3%   15.2%   10.4%    7.3%  (n=441)
   neutral   10.6%    7.8%    0.9%   15.5%   44.6%    7.3%   13.3%  (n=451)
       sad   15.2%    7.4%    1.1%   19.6%   14.1%   33.3%    9.3%  (n=270)
  surprise   15.9%    9.1%    0.5%   19.7%   28.8%   16.3%    9.6%  (n=208)

Global prior: ['angry:18.8%', 'disgust:9.4%', 'fear:1.4%', 'happy:23.3%', 'neutral:22.5%', 'sad:14.7%', 'surprise:9.8%']


In [5]:
class TransitionLabelSmoothing(nn.Module):
    """SL1: Smooth target toward empirical transition distribution instead of uniform.
    
    Standard label smoothing: target = (1-α)*one_hot + α*uniform
    Ours: target = (1-α)*one_hot + α*T[a_emotion]
    """
    def __init__(self, trans_matrix, alpha=0.2):
        super().__init__()
        self.register_buffer('T', trans_matrix)
        self.alpha = alpha
    
    def forward(self, logits, targets, a_emotions):
        """
        logits: [B, 7]
        targets: [B] (B's emotion, ground truth)
        a_emotions: [B] (A's emotion, from clip III)
        """
        B, C = logits.shape
        
        # Build soft targets
        one_hot = F.one_hot(targets, C).float()  # [B, C]
        
        # Get transition prior for each sample based on A's emotion
        trans_prior = self.T[a_emotions]  # [B, C]
        
        # Smooth: blend one-hot with transition prior
        soft_targets = (1 - self.alpha) * one_hot + self.alpha * trans_prior
        
        # KL divergence loss
        log_probs = F.log_softmax(logits, dim=-1)
        loss = -(soft_targets * log_probs).sum(dim=-1).mean()
        
        return loss
 
 
class CostSensitiveCE(nn.Module):
    """SL2: Penalize implausible transitions more than plausible ones.
    
    Cost of predicting class j when truth is k, given A's emotion i:
    cost[j] = 1 - T[i][j]  (rare transitions → high cost)
    """
    def __init__(self, trans_matrix, gamma=1.0):
        super().__init__()
        self.register_buffer('T', trans_matrix)
        self.gamma = gamma
    
    def forward(self, logits, targets, a_emotions):
        B, C = logits.shape
        
        # Standard CE per sample
        ce_per_sample = F.cross_entropy(logits, targets, reduction='none')  # [B]
        
        # Cost weight: how implausible is the TRUE class given A's emotion?
        # If T[a][true_b] is high → plausible → lower weight (model expected this)
        # If T[a][true_b] is low → implausible → higher weight (model should learn this)
        trans_probs = self.T[a_emotions]  # [B, C]
        true_trans_prob = trans_probs[torch.arange(B, device=logits.device), targets]  # [B]
        
        # Inverse frequency weighting: rare transitions get boosted
        weights = (1 - true_trans_prob) ** self.gamma + 0.5  # [B], minimum 0.5
        
        loss = (weights * ce_per_sample).mean()
        return loss
 
 
class TransitionPriorRegularization(nn.Module):
    """SL3: Blend model output with transition prior at training time.
    
    final_probs = (1-β)*model_probs + β*T[a_emotion]
    Loss computed on blended output.
    """
    def __init__(self, trans_matrix, beta=0.2):
        super().__init__()
        self.register_buffer('T', trans_matrix)
        self.beta = beta
    
    def forward(self, logits, targets, a_emotions):
        B, C = logits.shape
        
        model_probs = F.softmax(logits, dim=-1)  # [B, C]
        trans_prior = self.T[a_emotions]           # [B, C]
        
        # Blend
        blended = (1 - self.beta) * model_probs + self.beta * trans_prior  # [B, C]
        
        # NLL on blended probabilities
        log_blended = torch.log(blended + 1e-8)
        loss = F.nll_loss(log_blended, targets)
        
        return loss

In [6]:
class HiEFDataset(Dataset):
    def __init__(self, mcis_index, features_dir, indices):
        self.mcis_index = mcis_index
        self.features_dir = features_dir
        self.indices = indices
    def __len__(self): return len(self.indices)
    def _load(self, ff):
        d = torch.load(os.path.join(self.features_dir, ff), map_location='cpu', weights_only=False)
        return d['face_features'], d['ori_features'], d['text_feature'], d.get('audio_feature', torch.zeros(527))
    def __getitem__(self, idx):
        e = self.mcis_index[self.indices[idx]]
        c1f,c1o,c1t,c1a = self._load(e['feature_files'][0])
        c2f,c2o,c2t,c2a = self._load(e['feature_files'][1])
        c3f,c3o,c3t,c3a = self._load(e['feature_files'][2])
        return {
            'clip1_face':c1f,'clip1_ori':c1o,'clip1_text':c1t,'clip1_audio':c1a,
            'clip2_face':c2f,'clip2_ori':c2o,'clip2_text':c2t,'clip2_audio':c2a,
            'clip3_face':c3f,'clip3_ori':c3o,'clip3_text':c3t,'clip3_audio':c3a,
            'target':e['clip4_emotion'],
            'clip3_emotion':e['clip3_emotion'],
            'clip3_scene':e.get('clip3_scene',-1),
        }
 
def collate_fn(batch):
    r = {}
    for k in [f'clip{c}_{m}' for c in [1,2,3] for m in ['face','ori','text','audio']]:
        r[k] = torch.stack([b[k] for b in batch])
    for k in ['target','clip3_emotion','clip3_scene']:
        r[k] = torch.tensor([b[k] for b in batch], dtype=torch.long)
    return r
 
BATCH_SIZE = 32
train_loader = DataLoader(HiEFDataset(mcis_index, FEATURES_DIR, train_idx), batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn, num_workers=2, pin_memory=True)
val_loader = DataLoader(HiEFDataset(mcis_index, FEATURES_DIR, val_idx), batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=2, pin_memory=True)
test_loader = DataLoader(HiEFDataset(mcis_index, FEATURES_DIR, test_idx), batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=2, pin_memory=True)
 
# %%
# Model building blocks (same as Gap 1 notebook)
class TemporalTransformer(nn.Module):
    def __init__(self, d_model=512, n_heads=8, n_layers=2, dropout=0.1):
        super().__init__()
        self.pos_encoding = nn.Parameter(torch.randn(1, 16, d_model) * 0.02)
        el = nn.TransformerEncoderLayer(d_model, n_heads, d_model*4, dropout, batch_first=True, norm_first=True)
        self.transformer = nn.TransformerEncoder(el, num_layers=n_layers)
    def forward(self, x):
        return self.transformer(x + self.pos_encoding[:, :x.size(1), :])
 
class CrossAttentionFusion(nn.Module):
    def __init__(self, d_model=512, n_heads=8, n_layers=1, dropout=0.1):
        super().__init__()
        self.layers = nn.ModuleList([nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True) for _ in range(n_layers)])
        self.norms = nn.ModuleList([nn.LayerNorm(d_model) for _ in range(n_layers)])
    def forward(self, query, kv):
        x = query
        for attn, norm in zip(self.layers, self.norms):
            out, _ = attn(x, kv, kv)
            x = norm(x + out)
        return x
 
class IntraEncoder(nn.Module):
    def __init__(self, d_model=512, audio_dim=527, query_modality='face'):
        super().__init__()
        self.query_modality = query_modality
        self.face_temporal = TemporalTransformer(d_model)
        self.ori_temporal = TemporalTransformer(d_model)
        self.type_fusion = CrossAttentionFusion(d_model)
        self.audio_proj = nn.Linear(audio_dim, d_model)
        self.modality_fusion = CrossAttentionFusion(d_model)
    def forward(self, face, ori, text, audio):
        face_out = self.face_temporal(face).mean(dim=1, keepdim=True)
        ori_out = self.ori_temporal(ori).mean(dim=1, keepdim=True)
        vis = torch.cat([face_out, ori_out], dim=1)
        video_feat = self.type_fusion(face_out, vis)
        audio_feat = self.audio_proj(F.normalize(audio, dim=-1)).unsqueeze(1)
        text_feat = text.unsqueeze(1)
        mod_stack = torch.cat([video_feat, text_feat, audio_feat], dim=1)
        query = text_feat if self.query_modality == 'text' else video_feat
        return self.modality_fusion(query, mod_stack).squeeze(1)

In [7]:
class BaselineModel(nn.Module):
    """Shared encoder + LSTM+Transformer. Returns logits + clip3_emotion for loss."""
    def __init__(self, d=512, n_classes=7):
        super().__init__()
        self.encoder = IntraEncoder(d, query_modality='face')
        self.lstm = nn.LSTM(d, d, num_layers=3, batch_first=False, dropout=0.1)
        self.pos = nn.Parameter(torch.randn(1, 3, d) * 0.02)
        el = nn.TransformerEncoderLayer(d, 8, d*4, 0.1, batch_first=True, norm_first=True)
        self.trans = nn.TransformerEncoder(el, num_layers=2)
        self.head = nn.Sequential(nn.LayerNorm(d), nn.Dropout(0.3), nn.Linear(d,d//2), nn.GELU(), nn.Dropout(0.2), nn.Linear(d//2, n_classes))
    def forward(self, batch):
        f1 = self.encoder(batch['clip1_face'], batch['clip1_ori'], batch['clip1_text'], batch['clip1_audio'])
        f2 = self.encoder(batch['clip2_face'], batch['clip2_ori'], batch['clip2_text'], batch['clip2_audio'])
        f3 = self.encoder(batch['clip3_face'], batch['clip3_ori'], batch['clip3_text'], batch['clip3_audio'])
        seq = torch.stack([f1,f2,f3], dim=0)
        out, _ = self.lstm(seq)
        out = self.trans(out.permute(1,0,2) + self.pos).mean(dim=1)
        return self.head(out)
 
 
class Gap1v3Model(nn.Module):
    """Disentangled encoder + Option B fusion + auxiliary losses."""
    def __init__(self, d=512, n_classes=7, n_scenes=4):
        super().__init__()
        self.ctx_encoder = IntraEncoder(d, query_modality='text')
        self.emo_encoder = IntraEncoder(d, query_modality='face')
        self.ctx_fusion = CrossAttentionFusion(d)
        self.forecast = CrossAttentionFusion(d, n_layers=2)
        self.head = nn.Sequential(nn.LayerNorm(d), nn.Dropout(0.3), nn.Linear(d,d//2), nn.GELU(), nn.Dropout(0.2), nn.Linear(d//2, n_classes))
        self.scene_head = nn.Linear(d, n_scenes)
        self.emo_aux_head = nn.Linear(d, n_classes)
    
    def forward(self, batch):
        ctx1 = self.ctx_encoder(batch['clip1_face'], batch['clip1_ori'], batch['clip1_text'], batch['clip1_audio'])
        ctx2 = self.ctx_encoder(batch['clip2_face'], batch['clip2_ori'], batch['clip2_text'], batch['clip2_audio'])
        emo3 = self.emo_encoder(batch['clip3_face'], batch['clip3_ori'], batch['clip3_text'], batch['clip3_audio'])
        
        ctx_stack = torch.stack([ctx1, ctx2], dim=1)
        ctx_combined = self.ctx_fusion(ctx1.unsqueeze(1), ctx_stack).squeeze(1)
        
        emo_q = emo3.unsqueeze(1)
        ef_stack = torch.cat([emo_q, ctx_combined.unsqueeze(1)], dim=1)
        final = self.forecast(emo_q, ef_stack).squeeze(1)
        
        aux = {
            'scene_logits': self.scene_head(ctx_combined),
            'emo_a_logits': self.emo_aux_head(emo3),
        }
        return self.head(final), aux

In [8]:
def compute_metrics(preds, labels, n_classes=7):
    preds, labels = np.array(preds), np.array(labels)
    war = (preds == labels).sum() / len(labels) * 100
    recalls = []
    for c in range(n_classes):
        mask = labels == c
        if mask.sum() > 0: recalls.append((preds[mask] == c).sum() / mask.sum() * 100)
    return war, np.mean(recalls) if recalls else 0.0
 
def train_experiment(model, loss_fn, train_loader, val_loader, test_loader,
                     n_epochs=50, lr=1e-4, model_name="model", 
                     use_gap1v3_aux=False, aux_weight=0.3, device=DEVICE):
    """Unified training loop for all experiments."""
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)
    
    best_val_uar = 0; best_epoch = 0
    
    for epoch in range(1, n_epochs + 1):
        model.train()
        for batch in train_loader:
            bg = {k: v.to(device) for k, v in batch.items()}
            
            if use_gap1v3_aux:
                logits, aux = model(bg)
            else:
                logits = model(bg)
                aux = {}
            
            # Main loss (structured or flat CE)
            if loss_fn is not None:
                main_loss = loss_fn(logits, bg['target'], bg['clip3_emotion'])
            else:
                main_loss = F.cross_entropy(logits, bg['target'])
            
            # Auxiliary losses (Gap1v3 only)
            total_loss = main_loss
            if use_gap1v3_aux:
                if 'scene_logits' in aux:
                    mask = bg['clip3_scene'] >= 0
                    if mask.sum() > 0:
                        total_loss += aux_weight * F.cross_entropy(aux['scene_logits'][mask], bg['clip3_scene'][mask])
                if 'emo_a_logits' in aux:
                    mask = bg['clip3_emotion'] >= 0
                    if mask.sum() > 0:
                        total_loss += aux_weight * F.cross_entropy(aux['emo_a_logits'][mask], bg['clip3_emotion'][mask])
            
            optimizer.zero_grad(); total_loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        
        # Val
        model.eval(); vp, vl = [], []; vl_total = 0
        with torch.no_grad():
            for batch in val_loader:
                bg = {k: v.to(device) for k, v in batch.items()}
                if use_gap1v3_aux:
                    logits, _ = model(bg)
                else:
                    logits = model(bg)
                vl_total += F.cross_entropy(logits, bg['target']).item() * logits.size(0)
                vp.extend(logits.argmax(-1).cpu().numpy())
                vl.extend(batch['target'].numpy())
        
        scheduler.step(vl_total / len(val_loader.dataset))
        vw, vu = compute_metrics(vp, vl)
        if vu > best_val_uar:
            best_val_uar = vu; best_epoch = epoch
            torch.save(model.state_dict(), f'/kaggle/working/{model_name}_best.pt')
        
        if epoch % 10 == 0 or epoch == 1:
            print(f"  [{model_name}] Ep {epoch:>3}: Val WAR={vw:.1f}%, UAR={vu:.1f}%"
                  f"{'  ★' if epoch == best_epoch else ''}")
    
    # Test
    model.load_state_dict(torch.load(f'/kaggle/working/{model_name}_best.pt', map_location=device, weights_only=True))
    model.eval(); tp, tl = [], []
    with torch.no_grad():
        for batch in test_loader:
            bg = {k: v.to(device) for k, v in batch.items()}
            if use_gap1v3_aux:
                logits, _ = model(bg)
            else:
                logits = model(bg)
            tp.extend(logits.argmax(-1).cpu().numpy())
            tl.extend(batch['target'].numpy())
    
    tw, tu = compute_metrics(tp, tl)
    tp_arr, tl_arr = np.array(tp), np.array(tl)
    
    per_class = {}
    for c in range(7):
        mask = tl_arr == c
        per_class[emo_names[c]] = (tp_arr[mask] == c).sum() / mask.sum() * 100 if mask.sum() > 0 else 0.0
    
    return {
        'model_name': model_name, 'best_epoch': best_epoch, 'best_val_uar': best_val_uar,
        'test_war': tw, 'test_uar': tu, 'per_class': per_class,
        'n_predicted_classes': len(set(tp_arr)),
        'test_preds': tp_arr, 'test_labels': tl_arr,
    }

In [9]:
print("=" * 70)
print("Gap 3: Structured Loss Experiments (50 epochs each)")
print("=" * 70)
results = {}
 
# --- Exp 1: Baseline + flat CE ---
print("\n[1/7] Baseline + flat CE")
m = BaselineModel().to(DEVICE)
results['baseline_ce'] = train_experiment(m, None, train_loader, val_loader, test_loader, model_name="baseline_ce")
 
# --- Exp 2: Baseline + SL1 (transition label smoothing) ---
print("\n[2/7] Baseline + SL1 (transition label smoothing, α=0.2)")
m = BaselineModel().to(DEVICE)
sl1 = TransitionLabelSmoothing(T_MATRIX, alpha=0.2)
results['baseline_sl1'] = train_experiment(m, sl1, train_loader, val_loader, test_loader, model_name="baseline_sl1")
 
# --- Exp 3: Baseline + SL2 (cost-sensitive CE) ---
print("\n[3/7] Baseline + SL2 (cost-sensitive CE, γ=1.0)")
m = BaselineModel().to(DEVICE)
sl2 = CostSensitiveCE(T_MATRIX, gamma=1.0)
results['baseline_sl2'] = train_experiment(m, sl2, train_loader, val_loader, test_loader, model_name="baseline_sl2")
 
# --- Exp 4: Baseline + SL3 (transition prior regularization) ---
print("\n[4/7] Baseline + SL3 (transition prior, β=0.2)")
m = BaselineModel().to(DEVICE)
sl3 = TransitionPriorRegularization(T_MATRIX, beta=0.2)
results['baseline_sl3'] = train_experiment(m, sl3, train_loader, val_loader, test_loader, model_name="baseline_sl3")
 
# --- Exp 5: Gap1v3 + flat CE ---
print("\n[5/7] Gap1v3 + flat CE")
m = Gap1v3Model(n_scenes=len(scene_map)).to(DEVICE)
results['gap1v3_ce'] = train_experiment(m, None, train_loader, val_loader, test_loader, 
                                         model_name="gap1v3_ce", use_gap1v3_aux=True)
 
# --- Exp 6: Gap1v3 + best SL (will pick after seeing exp 2-4) ---
# For now, run all 3 SLs with Gap1v3
 
print("\n[6/7] Gap1v3 + SL1 (transition label smoothing)")
m = Gap1v3Model(n_scenes=len(scene_map)).to(DEVICE)
sl1 = TransitionLabelSmoothing(T_MATRIX, alpha=0.2)
results['gap1v3_sl1'] = train_experiment(m, sl1, train_loader, val_loader, test_loader,
                                          model_name="gap1v3_sl1", use_gap1v3_aux=True)
 
print("\n[7/7] Gap1v3 + SL3 (transition prior)")
m = Gap1v3Model(n_scenes=len(scene_map)).to(DEVICE)
sl3 = TransitionPriorRegularization(T_MATRIX, beta=0.2)
results['gap1v3_sl3'] = train_experiment(m, sl3, train_loader, val_loader, test_loader,
                                          model_name="gap1v3_sl3", use_gap1v3_aux=True)

Gap 3: Structured Loss Experiments (50 epochs each)

[1/7] Baseline + flat CE


/tmp/ipykernel_58/1484784389.py:44: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(el, num_layers=n_layers)
/tmp/ipykernel_58/2803236262.py:9: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.trans = nn.TransformerEncoder(el, num_layers=2)


  [baseline_ce] Ep   1: Val WAR=24.1%, UAR=14.3%  ★
  [baseline_ce] Ep  10: Val WAR=39.2%, UAR=25.8%  ★
  [baseline_ce] Ep  20: Val WAR=37.5%, UAR=26.0%
  [baseline_ce] Ep  30: Val WAR=33.3%, UAR=24.3%
  [baseline_ce] Ep  40: Val WAR=31.6%, UAR=23.3%
  [baseline_ce] Ep  50: Val WAR=31.4%, UAR=23.6%

[2/7] Baseline + SL1 (transition label smoothing, α=0.2)


/tmp/ipykernel_58/1484784389.py:44: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(el, num_layers=n_layers)
/tmp/ipykernel_58/2803236262.py:9: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.trans = nn.TransformerEncoder(el, num_layers=2)


  [baseline_sl1] Ep   1: Val WAR=24.1%, UAR=14.3%  ★
  [baseline_sl1] Ep  10: Val WAR=33.7%, UAR=22.4%
  [baseline_sl1] Ep  20: Val WAR=38.2%, UAR=25.1%
  [baseline_sl1] Ep  30: Val WAR=35.4%, UAR=24.8%
  [baseline_sl1] Ep  40: Val WAR=34.7%, UAR=24.8%
  [baseline_sl1] Ep  50: Val WAR=34.2%, UAR=24.3%

[3/7] Baseline + SL2 (cost-sensitive CE, γ=1.0)
  [baseline_sl2] Ep   1: Val WAR=23.1%, UAR=14.3%  ★
  [baseline_sl2] Ep  10: Val WAR=35.8%, UAR=24.0%
  [baseline_sl2] Ep  20: Val WAR=35.1%, UAR=23.8%
  [baseline_sl2] Ep  30: Val WAR=34.0%, UAR=24.3%
  [baseline_sl2] Ep  40: Val WAR=33.0%, UAR=23.9%
  [baseline_sl2] Ep  50: Val WAR=32.3%, UAR=23.4%

[4/7] Baseline + SL3 (transition prior, β=0.2)
  [baseline_sl3] Ep   1: Val WAR=24.1%, UAR=14.3%  ★
  [baseline_sl3] Ep  10: Val WAR=36.6%, UAR=24.4%  ★
  [baseline_sl3] Ep  20: Val WAR=35.8%, UAR=24.1%
  [baseline_sl3] Ep  30: Val WAR=34.2%, UAR=23.7%
  [baseline_sl3] Ep  40: Val WAR=33.7%, UAR=23.5%
  [baseline_sl3] Ep  50: Val WAR=34.2%, U

In [10]:
print("\n" + "=" * 90)
print("RESULTS COMPARISON")
print("=" * 90)
 
exp_order = ['baseline_ce', 'baseline_sl1', 'baseline_sl2', 'baseline_sl3', 
             'gap1v3_ce', 'gap1v3_sl1', 'gap1v3_sl3']
 
print(f"\n{'Model':<25} | {'Best Ep':>7} | {'Val UAR':>7} | {'Test WAR':>8} | {'Test UAR':>8} | {'Classes':>7}")
print("-" * 80)
 
for name in exp_order:
    r = results[name]
    print(f"{r['model_name']:<25} | {r['best_epoch']:>7} | {r['best_val_uar']:>6.1f}% | "
          f"{r['test_war']:>7.1f}% | {r['test_uar']:>7.1f}% | {r['n_predicted_classes']:>3}/7")
 
# Per-class recall
print(f"\n{'Model':<25} |", end="")
for e in emo_names:
    print(f" {e[:7]:>7}", end="")
print()
print("-" * 82)
 
for name in exp_order:
    r = results[name]
    print(f"{r['model_name']:<25} |", end="")
    for e in emo_names:
        val = r['per_class'].get(e, 0)
        print(f" {val:>5.1f}%" if val > 0 else f"   0.0%✗", end="")
    print()


RESULTS COMPARISON

Model                     | Best Ep | Val UAR | Test WAR | Test UAR | Classes
--------------------------------------------------------------------------------
baseline_ce               |      19 |   27.2% |    37.1% |    25.7% |   5/7
baseline_sl1              |      21 |   28.7% |    39.9% |    27.1% |   4/7
baseline_sl2              |      26 |   26.0% |    37.6% |    26.1% |   6/7
baseline_sl3              |      25 |   25.0% |    34.7% |    24.7% |   5/7
gap1v3_ce                 |      18 |   27.8% |    33.1% |    25.3% |   7/7
gap1v3_sl1                |      27 |   29.2% |    35.0% |    25.5% |   7/7
gap1v3_sl3                |       8 |   26.4% |    31.9% |    23.2% |   7/7

Model                     |   angry disgust    fear   happy neutral     sad surpris
----------------------------------------------------------------------------------
baseline_ce               |  34.2%   0.0%✗   0.0%✗  53.5%  53.7%  32.8%   5.6%
baseline_sl1              |  51.3%   0.0%

In [11]:
print("\n" + "=" * 70)
print("TRANSITION ANALYSIS: Baseline CE vs Best Structured Loss")
print("=" * 70)
 
# Find best structured loss
best_sl_name = max(['baseline_sl1', 'baseline_sl2', 'baseline_sl3'], 
                    key=lambda n: results[n]['test_uar'])
print(f"\nBest structured loss (baseline): {best_sl_name} (UAR={results[best_sl_name]['test_uar']:.1f}%)")
 
for name in ['baseline_ce', best_sl_name]:
    r = results[name]
    pred_trans = np.zeros((7, 7))
    for i, idx in enumerate(test_idx):
        a = mcis_index[idx]['clip3_emotion']
        if a >= 0:
            pred_trans[a][r['test_preds'][i]] += 1
    
    ps = pred_trans.sum(axis=1, keepdims=True); ps[ps==0] = 1
    pred_prob = pred_trans / ps
    
    print(f"\n--- {name} predicted transitions ---")
    header = f"{'A\\B':>10}" + "".join(f"{emo_names[j][:7]:>8}" for j in range(7))
    print(header)
    for i in range(7):
        row = f"{emo_names[i]:>10}"
        for j in range(7):
            row += f"{pred_prob[i][j]*100:>7.1f}%"
        print(row)


TRANSITION ANALYSIS: Baseline CE vs Best Structured Loss

Best structured loss (baseline): baseline_sl1 (UAR=27.1%)

--- baseline_ce predicted transitions ---
       A\B   angry disgust    fear   happy neutral     sad surpris
     angry   30.8%    0.0%    0.0%   17.6%   12.1%   27.5%   12.1%
   disgust   22.5%    0.0%    0.0%   37.5%   22.5%   17.5%    0.0%
      fear   50.0%    0.0%    0.0%    0.0%   12.5%   37.5%    0.0%
     happy   10.6%    0.0%    0.0%   54.1%   14.1%   18.8%    2.4%
   neutral    8.3%    0.0%    0.0%   21.9%   55.2%   10.4%    4.2%
       sad   25.0%    0.0%    0.0%   29.7%    9.4%   32.8%    3.1%
  surprise   14.3%    0.0%    0.0%   19.0%   31.0%   16.7%   19.0%

--- baseline_sl1 predicted transitions ---
       A\B   angry disgust    fear   happy neutral     sad surpris
     angry   47.3%    0.0%    0.0%   19.8%   22.0%   11.0%    0.0%
   disgust   40.0%    0.0%    0.0%   30.0%   25.0%    5.0%    0.0%
      fear   62.5%    0.0%    0.0%    0.0%   25.0%   12.5% 

In [12]:
print("\n" + "=" * 70)
print("SUMMARY")
print("=" * 70)
 
baseline_uar = results['baseline_ce']['test_uar']
print(f"\nBaseline (flat CE): WAR={results['baseline_ce']['test_war']:.1f}%, UAR={baseline_uar:.1f}%")
 
for name in exp_order[1:]:
    r = results[name]
    delta = r['test_uar'] - baseline_uar
    d = "↑" if delta > 0 else "↓"
    fear = r['per_class'].get('fear', 0)
    surprise = r['per_class'].get('surprise', 0)
    minority = f"fear={fear:.0f}%, sur={surprise:.0f}%"
    print(f"  {r['model_name']:<25}: UAR {d}{abs(delta):.1f}pts → {r['test_uar']:.1f}%  [{minority}]")
 
# Best overall
best_name = max(exp_order, key=lambda n: results[n]['test_uar'])
best = results[best_name]
print(f"\nBest overall: {best_name} — WAR={best['test_war']:.1f}%, UAR={best['test_uar']:.1f}%")
 
# Save
save_data = {}
for name in exp_order:
    r = results[name]
    save_data[name] = {k: v for k, v in r.items() if k not in ['test_preds', 'test_labels']}
 
with open('/kaggle/working/gap3_results.json', 'w') as f:
    json.dump(save_data, f, indent=2, default=str)
 
print("\nSaved to /kaggle/working/gap3_results.json")


SUMMARY

Baseline (flat CE): WAR=37.1%, UAR=25.7%
  baseline_sl1             : UAR ↑1.4pts → 27.1%  [fear=0%, sur=0%]
  baseline_sl2             : UAR ↑0.4pts → 26.1%  [fear=0%, sur=0%]
  baseline_sl3             : UAR ↓1.0pts → 24.7%  [fear=0%, sur=0%]
  gap1v3_ce                : UAR ↓0.4pts → 25.3%  [fear=0%, sur=19%]
  gap1v3_sl1               : UAR ↓0.2pts → 25.5%  [fear=0%, sur=14%]
  gap1v3_sl3               : UAR ↓2.5pts → 23.2%  [fear=0%, sur=17%]

Best overall: baseline_sl1 — WAR=39.9%, UAR=27.1%

Saved to /kaggle/working/gap3_results.json
